In [2]:
import pandas as pd
df=pd.read_csv('/content/patient_health_records.CSV')

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   patient_id      500 non-null    object 
 1   age             488 non-null    float64
 2   gender          490 non-null    object 
 3   region          485 non-null    object 
 4   bmi             482 non-null    float64
 5   blood_pressure  486 non-null    float64
 6   cholesterol     484 non-null    float64
 7   glucose         480 non-null    float64
 8   disease_risk    500 non-null    int64  
dtypes: float64(5), int64(1), object(3)
memory usage: 35.3+ KB


In [4]:
df.isnull().sum()

,0
patient_id,0
age,12
gender,10
region,15
bmi,18
blood_pressure,14
cholesterol,16
glucose,20
disease_risk,0


**PART A **

## HANDLING MISSING VALUES
#1. `IDENTIFY MISSING VALUES AND PROVIDE A SUMMARY REPORT ( PERCENTAGE PER COLUMNS)  `

In [5]:
#IDENTIFY MISSING VALUES
missing_count=df.isnull().sum()

#calsulate percentage
missing_percentage=(missing_count/len(df)*100)

#summary report

missing_report=pd.DataFrame({"missing values":missing_count,"missing percentage":missing_percentage.round(2)})
print(missing_report)


                missing values  missing percentage
patient_id                   0                 0.0
age                         12                 2.4
gender                      10                 2.0
region                      15                 3.0
bmi                         18                 3.6
blood_pressure              14                 2.8
cholesterol                 16                 3.2
glucose                     20                 4.0
disease_risk                 0                 0.0


# **2.apply the following imputation technique and compare results**

# 2.1 simple imputer (numerical) replace missing bmi with mean median

In [6]:
from sklearn.impute import SimpleImputer
print('Befor the imputation')
print(df.isnull().sum())

imputer = SimpleImputer(strategy="mean")
df[["bmi"]] = imputer.fit_transform(df[["bmi"]])
data=pd.DataFrame(df)
print("missing bmi after mean imputation:")
print(data.isnull().sum())

Befor the imputation
patient_id         0
age               12
gender            10
region            15
bmi               18
blood_pressure    14
cholesterol       16
glucose           20
disease_risk       0
dtype: int64
missing bmi after mean imputation:
patient_id         0
age               12
gender            10
region            15
bmi                0
blood_pressure    14
cholesterol       16
glucose           20
disease_risk       0
dtype: int64


In [7]:
imputer = SimpleImputer(strategy="median")
print('Befor the imputation')
print(df.isnull().sum())
df[["bmi"]] = imputer.fit_transform(df[["bmi"]])
data=pd.DataFrame(df)
print("missing bmi after median imputation:")
print(data.isnull().sum())

Befor the imputation
patient_id         0
age               12
gender            10
region            15
bmi                0
blood_pressure    14
cholesterol       16
glucose           20
disease_risk       0
dtype: int64
missing bmi after median imputation:
patient_id         0
age               12
gender            10
region            15
bmi                0
blood_pressure    14
cholesterol       16
glucose           20
disease_risk       0
dtype: int64


# 2.2 simple imputer (categorical) replace missing region with the most frequent value

In [8]:

imputer = SimpleImputer(strategy="most_frequent")
print('Befor the imputation')
print(df.isnull().sum())
df[["region"]] = imputer.fit_transform(df[["region"]])
data=pd.DataFrame(df)
print("missing region after  imputation:")
print(data.isnull().sum())

Befor the imputation
patient_id         0
age               12
gender            10
region            15
bmi                0
blood_pressure    14
cholesterol       16
glucose           20
disease_risk       0
dtype: int64
missing region after  imputation:
patient_id         0
age               12
gender            10
region             0
bmi                0
blood_pressure    14
cholesterol       16
glucose           20
disease_risk       0
dtype: int64


In [9]:
print("Most frequent region:", df["region"].mode()[0])

Most frequent region: South


In [10]:

imputer = SimpleImputer(strategy="most_frequent")
print('Befor the imputation')
print(df.isnull().sum())
df[["gender"]] = imputer.fit_transform(df[["gender"]])

print("missing gender after  imputation:")
print("gender",df.isnull().sum())

Befor the imputation
patient_id         0
age               12
gender            10
region             0
bmi                0
blood_pressure    14
cholesterol       16
glucose           20
disease_risk       0
dtype: int64
missing gender after  imputation:
gender patient_id         0
age               12
gender             0
region             0
bmi                0
blood_pressure    14
cholesterol       16
glucose           20
disease_risk       0
dtype: int64


In [11]:
print("Most frequent gender:", df["gender"].mode()[0])

Most frequent gender: Female


# 2.4 Missing Indicator + Random Sample Imputation: Create binary indicator columns for missingness and use random sampling to impute values

In [12]:
import numpy as np
df["bmi_missing"] = df["bmi"].isna().astype(int)
df.loc[df["bmi"].isna(), "bmi"] = np.random.choice(
    df["bmi"].dropna(),
    df["bmi"].isna().sum())
print("AFTER BMI AFTER IMPUTATION ",df.isnull().sum())



AFTER BMI AFTER IMPUTATION  patient_id         0
age               12
gender             0
region             0
bmi                0
blood_pressure    14
cholesterol       16
glucose           20
disease_risk       0
bmi_missing        0
dtype: int64


# 2.5 KNN Imputer: Apply k-Nearest Neighbors imputation for impute values  

In [13]:
from sklearn.impute import KNNImputer

cols = ["age", "bmi", "blood_pressure", "cholesterol", "glucose"]

imputer = KNNImputer(n_neighbors=5)
df[cols] = imputer.fit_transform(df[cols])

print("\nAFTER KNN IMPUTATION")
print(df[cols].isnull().sum())


AFTER KNN IMPUTATION
age               0
bmi               0
blood_pressure    0
cholesterol       0
glucose           0
dtype: int64


MIICE Algorithm: Perform chained equations imputation for multiple variables simultaneously.

In [14]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
df = pd.read_csv('/content/patient_health_records.CSV')
print("*Before imputation*")
print(df.isnull().sum())

cols2 = ['blood_pressure', 'cholesterol', 'glucose']

im = IterativeImputer(max_iter=10, random_state=0)

df[cols2] = im.fit_transform(df[cols2])

print("**After imputation**")
print(df.isnull().sum())

*Before imputation*
patient_id         0
age               12
gender            10
region            15
bmi               18
blood_pressure    14
cholesterol       16
glucose           20
disease_risk       0
dtype: int64
**After imputation**
patient_id         0
age               12
gender            10
region            15
bmi               18
blood_pressure     0
cholesterol        0
glucose            0
disease_risk       0
dtype: int64


## **Handling Outliers**
Detect and remove outliers
Z-score method: Identify patients with extreme cholestrol or glucose values.

In [15]:
mean=df['cholesterol'].mean()
Std=df['cholesterol'].std()
df['Zscore']=(df['cholesterol']-mean)/Std

dfz=df[df['Zscore'].abs()<=3].drop(columns= ['Zscore'])
df=df.drop(columns=['Zscore'])
print('shape with outlier',df.shape)
print('mean with outlier',df['cholesterol'].mean())
print('shape without outlier',dfz.shape)
print('mean without outlier',dfz['cholesterol'].mean())

shape with outlier (500, 9)
mean with outlier 196.01739874074954
shape without outlier (498, 9)
mean without outlier 195.16807102484893


IQR method: use interquartile range to detect unusual BMI values.

In [16]:
q1=df['bmi'].quantile(0.25)
q3=df['bmi'].quantile(0.75)
iqr=q3-q1
upl=q3+1.5*iqr
lwl=q1-1.5*iqr

outlier=(df['bmi']>upl) | (df['bmi']<lwl)
df_iq=df[~outlier]

print('shape with outlier',df.shape)
print('mean with outlier',df['bmi'].mean())
print('shape without outlier',df_iq.shape)
print('mean without outlier',df_iq['bmi'].mean())

shape with outlier (500, 9)
mean with outlier 27.08402489626556
shape without outlier (495, 9)
mean without outlier 26.957651991614256


**Perecentile method**: Cap values below 1st percentile and above 99th percentile.

In [17]:
up_limit=df['glucose'].quantile(0.95)
lo_limit=df['glucose'].quantile(0.05)
mask=(df['glucose']<lo_limit) | (df['glucose']>up_limit)
df_p=df[~mask]
print('shape with outlier',df.shape)
print('mean with outlier',df['glucose'].mean())
print('shape without outlier',df_p.shape)
print('mean without outlier',df_p['glucose'].mean())

shape with outlier (500, 9)
mean with outlier 106.47458717514552
shape without outlier (450, 9)
mean without outlier 106.07198575016166


# Apply Winsorization to cap extreme outliers instead of removing them.

In [18]:
df['cholesterol_w']=df['cholesterol'].clip(lower=lo_limit,upper=up_limit)

print('shape with outlier',df.shape)
print('mean with outlier',df['cholesterol'].mean())
print('shape without outlier',df.shape)
print('mean without outlier',df['cholesterol_w'].mean())

shape with outlier (500, 10)
mean with outlier 196.01739874074954
shape without outlier (500, 10)
mean without outlier 142.74683999999993


# Compare dataset shape and summary before vs after outlier treatment

In [19]:
data={'Method':['Z-score(cholesterol)','IQR(bmi)','Perecentile(glucose)','Winsorization(cholesterol)'],
      'mean_with outlier':[df['cholesterol'].mean(),df['bmi'].mean(),df['glucose'].mean(),df['cholesterol'].mean()],
      'mean without outlier':[dfz['cholesterol'].mean(),df_iq['bmi'].mean(),df_p['glucose'].mean(),df['cholesterol_w'].mean()],
      'STD_with outlier':[df['cholesterol'].std(),df['bmi'].std(),df['glucose'].std(),df['cholesterol'].std()],
      'STD_without outlier':[dfz['cholesterol'].std(),df_iq['bmi'].std(),df_p['glucose'].std(),df['cholesterol_w'].std()]}

compare=pd.DataFrame(data)
print('compare dataset shape and summary before vs after outlier treatment')
compare

compare dataset shape and summary before vs after outlier treatment


,Method,mean_with outlier,mean without outlier,STD_with outlier,STD_without outlier
0,Z-score(cholesterol),196.017399,195.168071,38.575454,36.223186
1,IQR(bmi),27.084025,26.957652,5.478047,5.166181
2,Perecentile(glucose),106.474587,106.071986,26.843400,18.836722
3,Winsorization(cholesterol),196.017399,142.746840,38.575454,6.115372


**Final clean Dataset**

Present the final cleaned dataset with

In [20]:
df.isnull().sum()


,0
patient_id,0
age,12
gender,10
region,15
bmi,18
blood_pressure,0
cholesterol,0
glucose,0
disease_risk,0
cholesterol_w,0


**All missing values treated appropriately**.

## All outliers handled using suitable methods.

# **Brief Repor**t
1. Most Effective Imputation Strategy

KNN Imputation was the most effective in your notebook for numerical data completeness. It was applied to Age, BMI, Blood Pressure, Cholesterol, and Glucose and reduced missing values in all five columns to 0.




2. Best Outlier Handling Method

Z-score filtering preserved the most data among the demonstrated outlier-removal methods.

Method	Records after treatment	Data retained
Z-score – Cholesterol	498/500	99.6%
IQR – BMI	495/500	99.0%
Percentile – Glucose	450/500	90.0%

For Z-score, cholesterol mean changed only from 196.02 → 195.17, while standard deviation decreased from 38.58 → 36.22, indicating that extreme observations were reduced without removing many records.

3. How Data Cleaning Improved Usability

Data cleaning:

Identified and quantified missing values.
Applied numerical and categorical imputation.
Reduced missing values in selected health variables to zero.
Detected and handled extreme observations.
Compared mean, standard deviation, and dataset size before/after treatment.